# 주택담보대출 월 상환액 계산기

대출 조건에 따른 원리금균등·원금균등·만기일시 상환 결과를 비교합니다.

In [ ]:
# @title 대출 조건을 입력하고 실행하세요
대출금액_억원 = 1.0  # @param {type:"number", min:0.01, step:0.1}
대출기간_년 = 30  # @param {type:"integer", min:1, step:1}
연이율_퍼센트 = 4.5  # @param {type:"number", min:0, step:0.1}

import pandas as pd
from IPython.display import HTML, Markdown, display

principal = float(대출금액_억원) * 100_000_000
years = int(대출기간_년)
annual_rate = float(연이율_퍼센트)

if principal <= 0:
    raise ValueError('대출금액은 0보다 커야 합니다.')
if years <= 0:
    raise ValueError('대출기간은 1년 이상이어야 합니다.')
if annual_rate < 0:
    raise ValueError('연이율은 0% 이상이어야 합니다.')

months = years * 12
monthly_rate = annual_rate / 100 / 12

def format_korean_currency(amount):
    amount = int(round(amount))
    if amount == 0:
        return '0원'
    eok, remainder = divmod(amount, 100_000_000)
    man, won = divmod(remainder, 10_000)
    parts = []
    if eok:
        parts.append(f'{eok:,}억')
    if man:
        parts.append(f'{man:,}만')
    if won:
        parts.append(f'{won:,}')
    return ' '.join(parts) + '원'

def equal_principal_and_interest():
    if monthly_rate == 0:
        monthly_payment = principal / months
    else:
        compound = (1 + monthly_rate) ** months
        monthly_payment = principal * monthly_rate * compound / (compound - 1)

    balance = principal
    total_interest = 0.0
    last_payment = monthly_payment
    annual_schedule = []
    yearly_payment = yearly_principal = yearly_interest = 0.0
    for month in range(1, months + 1):
        interest = balance * monthly_rate
        principal_payment = monthly_payment - interest
        actual_payment = monthly_payment
        if month == months:
            principal_payment = balance
            last_payment = principal_payment + interest
            actual_payment = last_payment
        balance -= principal_payment
        total_interest += interest
        yearly_payment += actual_payment
        yearly_principal += principal_payment
        yearly_interest += interest

        if month % 12 == 0:
            annual_schedule.append({
                '연차': month // 12,
                '연간 납입액': round(yearly_payment),
                '상환 원금': round(yearly_principal),
                '납부 이자': round(yearly_interest),
                '연말 잔액': round(max(balance, 0)),
            })
            yearly_payment = yearly_principal = yearly_interest = 0.0

    summary = {
        '상환 방식': '원리금균등상환',
        '첫 달 납입액': round(monthly_payment),
        '마지막 달 납입액': round(last_payment),
        '총 납부 이자': round(total_interest),
        '총 상환 금액': round(principal + total_interest),
    }
    return summary, annual_schedule

def equal_principal():
    monthly_principal = principal / months
    first_interest = principal * monthly_rate
    last_interest = monthly_principal * monthly_rate
    total_interest = monthly_rate * principal * (months + 1) / 2
    return {
        '상환 방식': '원금균등상환',
        '첫 달 납입액': round(monthly_principal + first_interest),
        '마지막 달 납입액': round(monthly_principal + last_interest),
        '총 납부 이자': round(total_interest),
        '총 상환 금액': round(principal + total_interest),
    }

def bullet_maturity():
    monthly_interest = principal * monthly_rate
    total_interest = monthly_interest * months
    return {
        '상환 방식': '만기일시상환',
        '첫 달 납입액': round(monthly_interest),
        '마지막 달 납입액': round(principal + monthly_interest),
        '총 납부 이자': round(total_interest),
        '총 상환 금액': round(principal + total_interest),
    }

equal_summary, annual_schedule = equal_principal_and_interest()
results = [
    equal_summary,
    equal_principal(),
    bullet_maturity(),
]
result_table = pd.DataFrame(results)
result_display = result_table.copy()
for column in result_display.columns[1:]:
    result_display[column] = result_display[column].map(
        format_korean_currency
    )

display(Markdown(
    f'## 계산 결과\n'
    f'- 대출금액: **{format_korean_currency(principal)}**\n'
    f'- 대출기간: **{years}년 ({months:,}개월)**\n'
    f'- 연이율: **{annual_rate:g}%**'
))
display(HTML(result_display.to_html(index=False)))

annual_table = pd.DataFrame(annual_schedule)
annual_display = annual_table.copy()
for column in annual_display.columns[1:]:
    annual_display[column] = annual_display[column].map(
        format_korean_currency
    )
display(Markdown('## 원리금균등상환 연도별 내역'))
display(HTML(annual_display.to_html(index=False)))

monthly_payment = results[0]['첫 달 납입액']
per_eok_payment = monthly_payment / 대출금액_억원
display(Markdown(
    f'### 한 줄 결론\n'
    f'**{대출금액_억원:g}억 원**을 **{years}년**, **연 {annual_rate:g}%**, '
    f'**원리금균등상환**으로 빌리면 월 상환액은 약 '
    f'**{format_korean_currency(monthly_payment)}**입니다.  '
    f'1억 원당 월 상환액으로 환산하면 약 '
    f'**{format_korean_currency(per_eok_payment)}**입니다.'
))